In [ ]:
!wget "https://drive.upm.es/s/TdqfDr25NAsGIea/download?path=%2F&files=test.zip"

--2023-05-24 20:43:23--  https://drive.upm.es/s/TdqfDr25NAsGIea/download?path=%2F&files=test.zip
Resolving drive.upm.es (drive.upm.es)... 138.100.4.11
Connecting to drive.upm.es (drive.upm.es)|138.100.4.11|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 250698837 (239M) [application/zip]
Saving to: ‘download?path=%2F&files=test.zip’

download?path=%2F&f 100%[===================>] 239.08M  2.99MB/s    in 80s     

2023-05-24 20:44:43 (2.99 MB/s) - ‘download?path=%2F&files=test.zip’ saved [250698837/250698837]



In [ ]:
!unzip dataset.zip

Archive:  dataset.zip
   creating: test/
   creating: test/gopro1/
   creating: test/gopro10/
  inflating: test/gopro10/groundtruth.txt  
   creating: test/gopro10/images/
  inflating: test/gopro10/images/101GOPRO-GOPR8182.JPG  
  inflating: test/gopro10/images/101GOPRO-GOPR8184.JPG  
  inflating: test/gopro10/images/101GOPRO-GOPR8187.JPG  
  inflating: test/gopro10/images/101GOPRO-GOPR8189.JPG  
  inflating: test/gopro10/images/101GOPRO-GOPR8191.JPG  
  inflating: test/gopro10/images/101GOPRO-GOPR8202.JPG  
  inflating: test/gopro10/images/101GOPRO-GOPR8207.JPG  
  inflating: test/gopro10/images/101GOPRO-GOPR8209.JPG  
  inflating: test/gopro10/images/101GOPRO-GOPR8210.JPG  
  inflating: test/gopro10/images/101GOPRO-GOPR8212.JPG  
  inflating: test/gopro10/images/101GOPRO-GOPR8215.JPG  
  inflating: test/gopro10/images/101GOPRO-GOPR8216.JPG  
  inflating: test/gopro10/images/101GOPRO-GOPR8223.JPG  
  inflating: test/gopro10/images/101GOPRO-GOPR8224.JPG  
  inflating: test/gopro10/imag

In [ ]:
!mkdir 'my_dir'
!mkdir my_dir/images
%cd /content/my_dir
open('labels.txt', 'w').close()

/content/my_dir


In [ ]:
%cd /content

/content


In [ ]:
import os
import shutil

# Set the source and destination directories
src_dir = '/content/test'
dst_dir = '/content/my_dir'

# Iterate over the subdirectories in the source directory
for subdir in os.listdir(src_dir):
    subdir_path = os.path.join(src_dir, subdir)
    if os.path.isdir(subdir_path):
        # Iterate over the files in the image subdirectory
        image_dir = os.path.join(subdir_path, 'images')
        for filename in os.listdir(image_dir):
            src_path = os.path.join(image_dir, filename)
            dst_path = os.path.join(dst_dir+'/images', filename)
            shutil.move(src_path, dst_path)
        
        # Read the label file and append its contents to the merged label file
        label_file = os.path.join(subdir_path, 'groundtruth.txt')
        with open(label_file, 'r') as f:
            label_contents = f.read().strip()
            with open(os.path.join(dst_dir, 'labels.txt'), 'a') as merged_file:
                merged_file.write(label_contents + '\n')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!zip /content/my_dir/labels.txt

	zip warning: missing end signature--probably not a zip file (did you
	zip warning: remember to use binary mode when you transferred it?)
	zip warning: (if you are trying to read a damaged archive try -F)

zip error: Zip file structure invalid (/content/my_dir/labels.txt)


In [ ]:
from tensorflow.keras.models import load_model
model=load_model('/content/drive/MyDrive/for10epoch_tuned_1.h5')

In [ ]:
coords= []
with open("/content/ahmed_nasser_last.txt", 'r') as f:
    for line in f:
        lin=line.split(":")[1]
        coords.append([int(x) for x in lin.split(",")])

In [ ]:
import numpy as np
import cv2
from PIL import Image, ImageTransform


# Define the path to the dataset directory and label file
dataset_path = '/content/my_dir'
label_filename = '/content/my_dir/labels.txt'

# Read the label file for each image and create a list of image filenames in the order they appear in the file
image_labels = {}
with open(os.path.join(dataset_path, label_filename)) as f:
    for line in f:
        line = line.strip().split()
        image_filename = line[0]
        image_labels[image_filename] = np.array([int(x) for x in line[1:]])

In [ ]:
# Define a function to get the cropped images for an image
def get_cropped_images(image_path, coords):
    cropped_images = []
    for coord in coords:
      im = Image.open(image_path)
      result2 = im.transform((150,150), ImageTransform.QuadTransform(coord))
      hh=np.array(result2)
      cropped_images.append(hh)
    return cropped_images

In [ ]:
# Loop through each image in the dataset
predicted_labels = []
for image_filename in image_labels.keys():
    labels=[]
    image_path = os.path.join(dataset_path, 'images', image_filename)
    # Get the cropped images and forward them through the model
    cropped_images = get_cropped_images(image_path, coords)
    cropped_images_array = np.array(cropped_images)
    labels = model.predict(cropped_images_array)
    labels=np.argmax(labels, axis=1)
    #labels = np.round(labels).astype(int).flatten()
    # Store the predicted labels for each image
    predicted_labels.append(labels)

1/1 [==============================] - 0s 106ms/step


In [ ]:
#Compute the accuracy of the model for each image
accuracies = []
list_all={}
for i in range(len(predicted_labels)):
    num_objects = len(coords)
    num_correct = 0
    list_image=[]
    for j in range(num_objects):
        if predicted_labels[i][j] == image_labels[list(image_labels.keys())[i]][j]:
            num_correct += 1
        else:
          list_image.append(j)    
    accuracy = num_correct / num_objects
    accuracies.append(accuracy)
    if list_image is not None and len(list_image) > 0: 
      list_all[list(image_labels.keys())[i]]=list_image
# Compute the overall accuracy of the model
overall_accuracy = np.mean(accuracies)
print('Overall Accuracy:', overall_accuracy)


Overall Accuracy: 0.957274459974587


In [ ]:
list_all

{'100GOPRO-GOPR7640.JPG': [14],
 '100GOPRO-GOPR6601.JPG': [14],
 '100GOPRO-GOPR7553.JPG': [14],
 '100GOPRO-GOPR6739.JPG': [6],
 '100GOPRO-GOPR6529.JPG': [14],
 '100GOPRO-GOPR7869.JPG': [14],
 '100GOPRO-GOPR7887.JPG': [14],
 '100GOPRO-GOPR6707.JPG': [14],
 '100GOPRO-GOPR6524.JPG': [14],
 '100GOPRO-GOPR6597.JPG': [14],
 '100GOPRO-GOPR6510.JPG': [14],
 '100GOPRO-GOPR7694.JPG': [14],
 '100GOPRO-GOPR6599.JPG': [14],
 '100GOPRO-GOPR7755.JPG': [14],
 '100GOPRO-GOPR6530.JPG': [14],
 '100GOPRO-GOPR6518.JPG': [14],
 '100GOPRO-GOPR7895.JPG': [14],
 '100GOPRO-GOPR6491.JPG': [14],
 '100GOPRO-GOPR7823.JPG': [14],
 '100GOPRO-GOPR6430.JPG': [14],
 '100GOPRO-GOPR7860.JPG': [14],
 '100GOPRO-GOPR6512.JPG': [14],
 '100GOPRO-GOPR7837.JPG': [14],
 '100GOPRO-GOPR6573.JPG': [14],
 '100GOPRO-GOPR6628.JPG': [14],
 '100GOPRO-GOPR6634.JPG': [14],
 '100GOPRO-GOPR6685.JPG': [14],
 '100GOPRO-GOPR7664.JPG': [14],
 '100GOPRO-GOPR7678.JPG': [14],
 '100GOPRO-GOPR7879.JPG': [14],
 '100GOPRO-GOPR7742.JPG': [14],
 '100GOPR

In [ ]:
list_imagess = []
for image_filename in list_all.keys():
    list_of_image = []
    image_path = os.path.join(dataset_path, 'images', image_filename)
    cropped_images = get_cropped_images(image_path, coords)
    for i, cropped_image in enumerate(cropped_images):
        if i in list_all[image_filename]:
            list_of_image.append(cropped_image)
    list_imagess.append(list_of_image)

In [ ]:
import os
import numpy as np

# Define the directory where you want to save the images
save_dir = '/content/Data_as'

# Loop through each image in the list
for i, image in enumerate(list_imagess):
    # Get the corresponding key from the dictionary
    image_filename = list(list_all.keys())[i]
    # Create a directory with the name of the key, if it doesn't exist
    #name = os.path.join(list(list_all.keys())[i], f"{image_filename.split('.')[0]}")
    name=f"{image_filename.split('.')[0]}"

    image_dir = os.path.join(save_dir, name)
    if not os.path.exists(image_dir):
        os.makedirs(image_dir)
    # Loop through each cropped image and save it
    for j, cropped_image in enumerate(image):
        # Define the name of the file based on the index
        filename = '{}_{}.png'.format(image_filename, j)
        new_name = filename.split(".")[0]+"-"+str(j) + ".JPG"

        # Save the cropped image to the directory
        save_path = os.path.join(image_dir, new_name)
        cropped_image = cv2.cvtColor(cropped_image, cv2.COLOR_RGB2BGR)
        #np.save(save_path, cropped_image)
        cv2.imwrite(save_path, cropped_image)

In [ ]:
!zip -r /content/Data_as.zip /content/Data_as

  adding: content/Data_as/ (stored 0%)
  adding: content/Data_as/107GOPRO-GOPR1652/ (stored 0%)
  adding: content/Data_as/107GOPRO-GOPR1652/107GOPRO-GOPR1652-0.JPG (deflated 1%)
  adding: content/Data_as/108GOPRO-GOPR7886/ (stored 0%)
  adding: content/Data_as/108GOPRO-GOPR7886/108GOPRO-GOPR7886-0.JPG (deflated 3%)
  adding: content/Data_as/108GOPRO-GOPR7886/108GOPRO-GOPR7886-1.JPG (deflated 2%)
  adding: content/Data_as/106GOPRO-GOPR2238/ (stored 0%)
  adding: content/Data_as/106GOPRO-GOPR2238/106GOPRO-GOPR2238-1.JPG (deflated 2%)
  adding: content/Data_as/106GOPRO-GOPR2238/106GOPRO-GOPR2238-0.JPG (deflated 3%)
  adding: content/Data_as/106GOPRO-GOPR2286/ (stored 0%)
  adding: content/Data_as/106GOPRO-GOPR2286/106GOPRO-GOPR2286-1.JPG (deflated 3%)
  adding: content/Data_as/106GOPRO-GOPR2286/106GOPRO-GOPR2286-0.JPG (deflated 3%)
  adding: content/Data_as/107GOPRO-GOPR7299/ (stored 0%)
  adding: content/Data_as/107GOPRO-GOPR7299/107GOPRO-GOPR7299-0.JPG (deflated 2%)
  adding: content/Da

In [ ]:
from google.colab import files
files.download("/content/Data_as.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
evaluate_CNR_EXT=model.evaluate( generate_data('/content/cnr-ext/test' ))
evaluate_pklot=model.evaluate( generate_data('/content/all_lots/test_lots' ))
evaluate_CNR=model.evaluate( generate_data('/content/cnr/test' ))

Found 28995 images belonging to 2 classes.


/usr/local/lib/python3.10/dist-packages/keras/preprocessing/image.py:1861: UserWarning: This ImageDataGenerator specifies `featurewise_center`, but it hasn't been fit on any training data. Fit it first by calling `.fit(numpy_data)`.
  warnings.warn(


227/227 [==============================] - 209s 917ms/step - loss: 0.0116 - accuracy: 0.9974
Found 139165 images belonging to 2 classes.
1088/1088 [==============================] - 992s 912ms/step - loss: 0.0039 - accuracy: 0.9993
Found 2516 images belonging to 2 classes.
20/20 [==============================] - 22s 1s/step - loss: 0.0052 - accuracy: 0.9980


In [ ]:
!zip -r /content/my_dir.zip /content/my_dir

	zip warning: missing end signature--probably not a zip file (did you
	zip warning: remember to use binary mode when you transferred it?)
	zip warning: (if you are trying to read a damaged archive try -F)

zip error: Zip file structure invalid (/content/my_dir/labels.txt)


In [ ]:
list_imagess_vacent = []
list_imagess_occupied = []

for image_filename in list_all.keys():
     
    image_path = os.path.join(dataset_path, 'images', image_filename)
    cropped_images = get_cropped_images(image_path, coords)

    for i, cropped_image in enumerate(cropped_images):

        if i in list_all[image_filename] and list(image_labels[image_filename])[i]==0:

            list_imagess_occupied.append(cropped_image)

        elif i in list_all[image_filename] and list(image_labels[image_filename])[i]==1:

            
            list_imagess_vacent.append(cropped_image)
           

In [ ]:
from PIL import Image

%cd /content/train_data/vacant

for i, arr in enumerate(list_imagess_vacent):
    
    img=Image.fromarray(arr)
    img.save(f"/content/drive/MyDrive/Untitled Folder 1/condation-img{i}.jpg")

   

/content/train_data/vacant


In [ ]:

%cd /content/train_data/occupied

for i, arr in enumerate(list_imagess_occupied):
    img=Image.fromarray(arr)
    img.save(f"/content/drive/MyDrive/Untitled Folder 2/condation-img{i}.jpg")
 

/content/train_data/occupied


In [ ]:
import matplotlib.pyplot as plt

print(len(list_imagess_vacent))

3733


In [ ]:
print(len(os.listdir('/content/train_data/occupied'))+len(os.listdir('/content/train_data/vacant')))

688933


In [ ]:
!wget http://www.inf.ufpr.br/vri/databases/PKLot.tar.gz
!tar xvzf  /content/PKLot.tar.gz

Streaming output truncated to the last 5000 lines.
PKLot/PKLotSegmented/PUC/Sunny/2012-09-20/Occupied/2012-09-20_18_09_45#088.jpg
PKLot/PKLotSegmented/PUC/Cloudy/2012-09-28/Occupied/2012-09-28_09_06_05#086.jpg
PKLot/PKLotSegmented/PUC/Cloudy/2012-10-31/Occupied/2012-10-31_14_33_21#092.jpg
PKLot/PKLotSegmented/UFPR04/Rainy/2013-01-21/Occupied/2013-01-21_08_35_04#011.jpg
PKLot/PKLotSegmented/PUC/Cloudy/2012-10-16/Empty/2012-10-16_06_26_44#050.jpg
PKLot/PKLotSegmented/PUC/Cloudy/2012-09-28/Occupied/2012-09-28_11_06_11#047.jpg
PKLot/PKLotSegmented/PUC/Sunny/2012-09-17/Occupied/2012-09-17_10_59_02#100.jpg
PKLot/PKLotSegmented/UFPR05/Sunny/2013-03-10/Empty/2013-03-10_17_35_13#027.jpg
PKLot/PKLotSegmented/PUC/Sunny/2012-10-27/Empty/2012-10-27_12_15_53#018.jpg
PKLot/PKLotSegmented/PUC/Rainy/2012-09-21/Occupied/2012-09-21_11_45_24#077.jpg
PKLot/PKLotSegmented/PUC/Sunny/2012-09-20/Occupied/2012-09-20_17_34_43#052.jpg
PKLot/PKLotSegmented/UFPR05/Cloudy/2013-03-17/Empty/2013-03-17_15_20_10#033.jpg

In [ ]:
!mkdir all_lots  all_lots/train_lots  all_lots/train_lots/occupied  all_lots/train_lots/vacant
!mkdir all_lots/test_lots  all_lots/test_lots/occupied all_lots/test_lots/vacant
#opereat on Empty or occupied parking space (division)
def split_data(Empty_directory_p,occupied_directory_p,path_test_vacant,path_test_occupied,path_train_vacant,path_train_occupied):
  import shutil , os
  from random import shuffle

  for empty  in Empty_directory_p:
      E=os.listdir(empty)
      shuffle(E)
      E1=round(len(E)*.80)
      empty_train=E[:E1]
      empty_test=E[E1:]

      for empty_Train_1  in empty_train:
        shutil.move(empty+'/'+empty_Train_1,path_train_vacant)
        
      for empty_test_1 in empty_test:
        shutil.move(empty+'/'+empty_test_1,path_test_vacant)

  #===================================================================================
  for occupied in occupied_directory_p:
    O=os.listdir(occupied)
    shuffle(O)
    O1=round(len(O)*.80)
    occupied_train=O[:O1]
    occupied_test=O[O1:]    

    for  occupied_train_1 in occupied_train : 
        shutil.move(occupied+'/'+occupied_train_1,path_train_occupied)

    for occupied_test_1 in occupied_test:
        shutil.move(occupied+'/'+occupied_test_1,path_test_occupied)

  #===========================================================================================



In [ ]:
#prepare pathes for data handling 
import os 
views=os.listdir('/content/PKLot/PKLotSegmented')
weathers=os.listdir('/content/PKLot/PKLotSegmented/PUC')

#===========================================================
directorys=[]
for view in views:
  for weather in weathers:
    directorys.append(os.path.join(view,weather))
%cd /content/PKLot/PKLotSegmented


Empty_directory=[]
Occupied_directory=[]
#==================================================================
for dir in directorys:
  temp_list=os.listdir(dir)
  for temp in temp_list:
    two_class=os.listdir(dir+'/'+temp)
    if len(two_class)==2:
      Empty_directory.append(os.path.join(dir,temp+'/Empty'))
      Occupied_directory.append(os.path.join(dir,temp+'/Occupied'))
    elif two_class[0]=='Empty':
      Empty_directory.append(os.path.join(dir,temp+'/Empty'))
    elif two_class[0]=='Occupied':
      Occupied_directory.append(os.path.join(dir,temp+'/Occupied'))

#======================================================================
#REMOVE TWO DIRECTORY THAT CONTAIN SINGLITON PHOTO
Empty_directory.remove('UFPR05/Rainy/2013-02-26/Empty')
Occupied_directory.remove('UFPR04/Rainy/2012-12-15/Occupied')
#=======================================================================

split_data(Empty_directory,Occupied_directory,'/content/all_lots/test_lots/vacant','/content/all_lots/test_lots/occupied',
           '/content/all_lots/train_lots/vacant','/content/all_lots/train_lots/occupied')


/content/PKLot/PKLotSegmented


In [ ]:
!mkdir /content/train_data  /content/train_data/vacant /content/train_data/occupied

In [ ]:
def move_to_final_train_or_test_file(path_to_file_data_O,path_to_file_data_v,path_to_file_O,path_to_file_v):
  dirs_o = os.listdir(path_to_file_data_O)
  dirs_v= os.listdir(path_to_file_data_v)
  import shutil
  os.chdir(path_to_file_data_O)

  for dir in dirs_o:
      shutil.move(dir,path_to_file_O)
  
  os.chdir(path_to_file_data_v)
  for dir in dirs_v:
    shutil.move(dir,path_to_file_v)
  os.chdir('/content')


#print( len(os.listdir('/content/train_data/vacant')) + len(os.listdir('/content/train_data/occupied')) )

In [ ]:
move_to_final_train_or_test_file('/content/all_lots/train_lots/occupied','/content/all_lots/train_lots/vacant',
                         '/content/train_data/occupied','/content/train_data/vacant')

In [ ]:
%cd /content
#download CNR-EXT dataset and unzip it 
!wget http://claudiotest.isti.cnr.it/park-datasets/CNR-EXT/CNR-EXT-Patches-150x150.zip
!unzip /content/CNR-EXT-Patches-150x150.zip

Streaming output truncated to the last 5000 lines.
  inflating: PATCHES/OVERCAST/2015-12-19/camera8/O_2015-12-19_11.06_C08_211.jpg  
  inflating: PATCHES/OVERCAST/2015-12-19/camera8/O_2015-12-19_08.06_C08_214.jpg  
  inflating: PATCHES/OVERCAST/2015-12-19/camera8/O_2015-12-19_10.36_C08_325.jpg  
  inflating: PATCHES/OVERCAST/2015-12-19/camera8/O_2015-12-19_16.36_C08_291.jpg  
  inflating: PATCHES/OVERCAST/2015-12-19/camera8/O_2015-12-19_10.06_C08_283.jpg  
  inflating: PATCHES/OVERCAST/2015-12-19/camera8/O_2015-12-19_08.36_C08_317.jpg  
  inflating: PATCHES/OVERCAST/2015-12-19/camera8/O_2015-12-19_15.06_C08_321.jpg  
  inflating: PATCHES/OVERCAST/2015-12-19/camera8/O_2015-12-19_11.36_C08_255.jpg  
  inflating: PATCHES/OVERCAST/2015-12-19/camera8/O_2015-12-19_09.36_C08_292.jpg  
  inflating: PATCHES/OVERCAST/2015-12-19/camera8/O_2015-12-19_11.36_C08_327.jpg  
  inflating: PATCHES/OVERCAST/2015-12-19/camera8/O_2015-12-19_16.06_C08_324.jpg  
  inflating: PATCHES/OVERCAST/2015-12-19/camera

In [ ]:
!mkdir CNR-EXT
!mkdir CNR-EXT/OverCast/
!mkdir CNR-EXT/Rainy/
!mkdir CNR-EXT/Sunnay/

In [ ]:
import os
dirs=os.listdir('/content/CNR-EXT')
%cd '/content/CNR-EXT'

for dir in dirs:
  for i in range(1,10):
      inner_path=os.path.join( dir,"C-{}".format(i) )
      os.mkdir(inner_path)
      vacent_path=os.path.join(inner_path,'vacent')
      occupied_path=os.path.join(inner_path,'occupied')
      os.mkdir(vacent_path)
      os.mkdir(occupied_path)
 


/content/CNR-EXT


In [ ]:
def move_data(path_label,number_of_camera):
  import shutil
  
  ref_to_file=open(path_label,'r')

  for line in ref_to_file:
    l=line.split(" ")

    if not l[0].find('SUNNY'):
      if  int(l[1])==1:
          shutil.move(line.split(" ")[0],'/content/CNR-EXT/Sunnay/C-{}/occupied'.format(number_of_camera))
      else: 
          shutil.move(line.split(" ")[0],'/content/CNR-EXT/Sunnay/C-{}/vacent'.format(number_of_camera))
    elif not l[0].find('RAINY'):
      if int(l[1])==1:
          shutil.move(line.split(" ")[0],'/content/CNR-EXT/Rainy/C-{}/occupied'.format(number_of_camera))
      else: 
          shutil.move(line.split(" ")[0],'/content/CNR-EXT/Rainy/C-{}/vacent'.format(number_of_camera))
    else:
      if int(l[1])==1:
          shutil.move(line.split(" ")[0],'/content/CNR-EXT/OverCast/C-{}/occupied'.format(number_of_camera))
      else: 
          shutil.move(line.split(" ")[0],'/content/CNR-EXT/OverCast/C-{}/vacent'.format(number_of_camera))



In [ ]:
path_data='/content/PATCHES'

%cd '/content/PATCHES'
for i in range(1,10):
  path_label='/content/LABELS/camera{}.txt'.format(i)
  print(path_label)  
  move_data(path_label,i)


/content/PATCHES
/content/LABELS/camera1.txt
/content/LABELS/camera2.txt
/content/LABELS/camera3.txt
/content/LABELS/camera4.txt
/content/LABELS/camera5.txt
/content/LABELS/camera6.txt
/content/LABELS/camera7.txt
/content/LABELS/camera8.txt
/content/LABELS/camera9.txt


In [ ]:
%cd /content

/content


In [ ]:
!mkdir cnr-ext
!mkdir cnr-ext/train cnr-ext/test
!mkdir cnr-ext/train/vacant cnr-ext/train/occupied 
!mkdir cnr-ext/test/vacant cnr-ext/test/occupied 

In [ ]:
dirs=os.listdir('/content/CNR-EXT')
%cd '/content/CNR-EXT'

Empty_directory_cnr=[]
occupied_directory_cnr=[]

for dir in dirs:
  for i in range(1,10):
      inner_path=os.path.join( dir,"C-{}".format(i) )
      vacent_path=os.path.join(inner_path,'vacent')
      occupied_path=os.path.join(inner_path,'occupied')
      Empty_directory_cnr.append('/content/CNR-EXT/'+vacent_path)
      occupied_directory_cnr.append('/content/CNR-EXT/'+occupied_path)   

/content/CNR-EXT


In [ ]:
split_data(Empty_directory_cnr,occupied_directory_cnr,'/content/cnr-ext/test/vacant','/content/cnr-ext/test/occupied',
           '/content/cnr-ext/train/vacant','/content/cnr-ext/train/occupied')

In [ ]:
move_to_final_train_or_test_file('/content/cnr-ext/train/occupied','/content/cnr-ext/train/vacant',
                         '/content/train_data/occupied','/content/train_data/vacant')

In [ ]:
#download CNR 
!wget http://claudiotest.isti.cnr.it/park-datasets/CNRPark/CNRPark-Patches-150x150.zip
!unzip /content/CNRPark-Patches-150x150.zip

Streaming output truncated to the last 5000 lines.
  inflating: B/busy/20150708_0855_1.jpg  
  inflating: B/busy/20150708_1620_25.jpg  
  inflating: B/busy/20150708_1240_21.jpg  
  inflating: B/busy/20150708_0950_48.jpg  
  inflating: B/busy/20150708_1705_4.jpg  
  inflating: B/busy/20150708_1400_21.jpg  
  inflating: B/busy/20150708_1300_14.jpg  
  inflating: B/busy/20150708_1625_11.jpg  
  inflating: B/busy/20150708_1505_50.jpg  
  inflating: B/busy/20150708_1720_15.jpg  
  inflating: B/busy/20150708_1640_45.jpg  
  inflating: B/busy/20150708_1105_42.jpg  
  inflating: B/busy/20150708_1150_35.jpg  
  inflating: B/busy/20150708_1230_53.jpg  
  inflating: B/busy/20150708_1255_38.jpg  
  inflating: B/busy/20150708_1140_3.jpg  
  inflating: B/busy/20150708_1140_31.jpg  
  inflating: B/busy/20150708_1040_22.jpg  
  inflating: B/busy/20150708_1140_6.jpg  
  inflating: B/busy/20150708_1150_41.jpg  
  inflating: B/busy/20150708_0945_47.jpg  
  inflating: B/busy/20150708_1105_50.jpg  
  infla

In [ ]:
!mkdir cnr
!mkdir cnr/test cnr/train 
!mkdir cnr/test/vacant cnr/test/occupied 
!mkdir cnr/train/vacant cnr/train/occupied 

In [ ]:
Empty_dir_cnr=['/content/A/free','/content/B/free']
Occupied_dir_cnr=['/content/A/busy','/content/B/busy']

split_data(Empty_dir_cnr,Occupied_dir_cnr,'/content/cnr/test/vacant',
           '/content/cnr/test/occupied','/content/cnr/train/vacant','/content/cnr/train/occupied')


In [ ]:
move_to_final_train_or_test_file('/content/cnr/train/occupied','/content/cnr/train/vacant',
                                 '/content/train_data/occupied','/content/train_data/vacant')

In [ ]:
print(len(os.listdir('/content/train_data/occupied'))+len(os.listdir('/content/train_data/vacant')))

682722


In [ ]:
from keras.preprocessing.image import ImageDataGenerator
def generate_data(path_to_data_dir):
    my_gen=ImageDataGenerator(1./255)
    
    data_gen=my_gen.flow_from_directory(
        path_to_data_dir,
        target_size=(150,150),
        batch_size=128,
        class_mode='categorical'
    )
    return data_gen

In [ ]:
history=model.fit_generator(
      generate_data('/content/train_data'),
      epochs=5,
      steps_per_epoch=688933/128 
      )

Found 688933 images belonging to 2 classes.


<ipython-input-56-7bac2d94e8fe>:1: UserWarning: `Model.fit_generator` is deprecated and will be removed in a future version. Please use `Model.fit`, which supports generators.
  history=model.fit_generator(
/usr/local/lib/python3.10/dist-packages/keras/preprocessing/image.py:1861: UserWarning: This ImageDataGenerator specifies `featurewise_center`, but it hasn't been fit on any training data. Fit it first by calling `.fit(numpy_data)`.
  warnings.warn(


Epoch 1/5
5382/5382 [==============================] - 1240s 230ms/step - loss: 0.0159 - accuracy: 0.9963
Epoch 2/5
5382/5382 [==============================] - 1064s 198ms/step - loss: 0.0090 - accuracy: 0.9979
Epoch 3/5
5382/5382 [==============================] - 981s 182ms/step - loss: 0.0081 - accuracy: 0.9982
Epoch 4/5
5382/5382 [==============================] - 907s 169ms/step - loss: 0.0080 - accuracy: 0.9983
Epoch 5/5
5382/5382 [==============================] - 857s 159ms/step - loss: 0.0069 - accuracy: 0.9984


In [ ]:
model.save('/content/drive/MyDrive/for10epoch_tuned.h5')

In [ ]:
history_1=model.fit_generator(
      generate_data('/content/train_data'),
      epochs=5,
      steps_per_epoch=688933/128 
      )

Found 688933 images belonging to 2 classes.


<ipython-input-63-d3d8e7a8d7d5>:1: UserWarning: `Model.fit_generator` is deprecated and will be removed in a future version. Please use `Model.fit`, which supports generators.
  history_1=model.fit_generator(
/usr/local/lib/python3.10/dist-packages/keras/preprocessing/image.py:1861: UserWarning: This ImageDataGenerator specifies `featurewise_center`, but it hasn't been fit on any training data. Fit it first by calling `.fit(numpy_data)`.
  warnings.warn(


Epoch 1/5
5382/5382 [==============================] - 1111s 206ms/step - loss: 0.0065 - accuracy: 0.9986
Epoch 2/5
5382/5382 [==============================] - 1089s 202ms/step - loss: 0.0071 - accuracy: 0.9985
Epoch 3/5
5382/5382 [==============================] - 1105s 205ms/step - loss: 0.0077 - accuracy: 0.9984
Epoch 4/5
5382/5382 [==============================] - 1065s 198ms/step - loss: 0.0055 - accuracy: 0.9988
Epoch 5/5
5382/5382 [==============================] - 1066s 198ms/step - loss: 0.0068 - accuracy: 0.9986


In [ ]:
model.save('/content/drive/MyDrive/for10epoch_tuned_1.h5')

In [ ]:
print(len(list_imagess_vacent))


3759


In [ ]:
print(len(list_imagess_occupied))

2922


In [ ]:
!mkdir "/content/drive/MyDrive/vacent" 

In [ ]:
!mkdir "/content/drive/MyDrive/occupied"

In [ ]:
import numpy as np


for i in list_imagess_vacent:
  np.save("/content/drive/MyDrive/vacent",i)

for i in list_imagess_occupied:
  np.save("/content/drive/MyDrive/occupied",i)

In [ ]:
import os
import numpy as np

# Define the directory where you want to save the images
save_dir = '/content/Data_as'

# Loop through each image in the list
for i, image in enumerate(list_imagess):
    # Get the corresponding key from the dictionary
    image_filename = list(list_all.keys())[i]
    # Create a directory with the name of the key, if it doesn't exist
    #name = os.path.join(list(list_all.keys())[i], f"{image_filename.split('.')[0]}")
    name=f"{image_filename.split('.')[0]}"

    image_dir = os.path.join(save_dir, name)
    if not os.path.exists(image_dir):
        os.makedirs(image_dir)
    # Loop through each cropped image and save it
    for j, cropped_image in enumerate(image):
        # Define the name of the file based on the index
        filename = '{}_{}.png'.format(image_filename, j)
        new_name = filename.split(".")[0]+"-"+str(j) + ".JPG"

        # Save the cropped image to the directory
        save_path = os.path.join(image_dir, new_name)
        cropped_image = cv2.cvtColor(cropped_image, cv2.COLOR_RGB2BGR)
        #np.save(save_path, cropped_image)
        cv2.imwrite(save_path, cropped_image)

In [ ]:
!zip -r /content/Data_as.zip /content/Data_as


Streaming output truncated to the last 5000 lines.
  adding: content/Data_as/108GOPRO-GOPR7388/108GOPRO-GOPR7388-0.JPG (deflated 1%)
  adding: content/Data_as/108GOPRO-GOPR7388/108GOPRO-GOPR7388-1.JPG (deflated 3%)
  adding: content/Data_as/100GOPRO-GOPR5466/ (stored 0%)
  adding: content/Data_as/100GOPRO-GOPR5466/100GOPRO-GOPR5466-1.JPG (deflated 2%)
  adding: content/Data_as/100GOPRO-GOPR5466/100GOPRO-GOPR5466-0.JPG (deflated 2%)
  adding: content/Data_as/107GOPRO-GOPR1647/ (stored 0%)
  adding: content/Data_as/107GOPRO-GOPR1647/107GOPRO-GOPR1647-0.JPG (deflated 1%)
  adding: content/Data_as/106GOPRO-GOPR2566/ (stored 0%)
  adding: content/Data_as/106GOPRO-GOPR2566/106GOPRO-GOPR2566-0.JPG (deflated 2%)
  adding: content/Data_as/106GOPRO-GOPR2566/106GOPRO-GOPR2566-1.JPG (deflated 1%)
  adding: content/Data_as/105GOPRO-GOPR6878/ (stored 0%)
  adding: content/Data_as/105GOPRO-GOPR6878/105GOPRO-GOPR6878-0.JPG (deflated 2%)
  adding: content/Data_as/100GOPRO-GOPR3684/ (stored 0%)
  adding

In [ ]:
from google.colab import files
files.download("/content/Data_as.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>